In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
import optuna
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Config
class CFG:
    train_path = "/kaggle/input/playground-series-s5e7/train.csv"
    test_path = "/kaggle/input/playground-series-s5e7/test.csv"
    original_path = "/kaggle/input/extrovert-vs-introvert-behavior-data/personality_datasert.csv"
    target = "Personality"
    n_folds = 5
    seed = 42
    n_trials = 100

In [3]:
# Load Data
train = pd.read_csv(CFG.train_path)
test = pd.read_csv(CFG.test_path)
original = pd.read_csv(CFG.original_path).rename(columns={'Personality': 'match_p'})
original = original.drop_duplicates(subset=original.columns.difference(['match_p']))

train = train.merge(original, how='left')
test = test.merge(original, how='left')

In [4]:
# Encode Target
label_encoder = LabelEncoder()
train[CFG.target] = label_encoder.fit_transform(train[CFG.target])

In [5]:
# Separate Features/Target
X = train.drop(columns=[CFG.target, "id"])
y = train[CFG.target]
X_test = test.drop(columns=["id"])

In [6]:
# Preprocessing Pipeline
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

In [7]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

In [8]:
# Optuna Objective
def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "use_label_encoder": False,
        "objective":"binary:hinge",
        "eval_metric": "auc",
        "random_state": CFG.seed,
        #"n_jobs": -1
    }

    clf = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(**params))
    ])

    skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
    scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        clf.fit(X_train, y_train)
        preds = clf.predict(X_val)
        score = accuracy_score(y_val, preds)
        scores.append(score)

    return np.mean(scores)

In [9]:
# Run Optuna Study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=CFG.n_trials)

print("Best trial:")
print(study.best_trial)

[I 2025-07-09 07:46:32,859] A new study created in memory with name: no-name-a3d22ec1-f4e9-4835-8a0c-4b6641c3bbd7
[I 2025-07-09 07:46:35,812] Trial 0 finished with value: 0.9615634846378281 and parameters: {'max_depth': 5, 'learning_rate': 0.29383111763879927, 'n_estimators': 383, 'subsample': 0.6358861764224504, 'colsample_bytree': 0.9632704064560585}. Best is trial 0 with value: 0.9615634846378281.
[I 2025-07-09 07:46:40,991] Trial 1 finished with value: 0.948067173249622 and parameters: {'max_depth': 10, 'learning_rate': 0.20509993096528198, 'n_estimators': 625, 'subsample': 0.8219241564130139, 'colsample_bytree': 0.8019889264716776}. Best is trial 0 with value: 0.9615634846378281.
[I 2025-07-09 07:46:45,721] Trial 2 finished with value: 0.9694991882430782 and parameters: {'max_depth': 6, 'learning_rate': 0.01853052363330515, 'n_estimators': 786, 'subsample': 0.855017876122148, 'colsample_bytree': 0.6194453269657145}. Best is trial 2 with value: 0.9694991882430782.
[I 2025-07-09 07:

Best trial:
FrozenTrial(number=36, state=1, values=[0.9699850182025924], datetime_start=datetime.datetime(2025, 7, 9, 7, 48, 54, 401072), datetime_complete=datetime.datetime(2025, 7, 9, 7, 48, 56, 866765), params={'max_depth': 3, 'learning_rate': 0.13032897007868877, 'n_estimators': 606, 'subsample': 0.694207201592242, 'colsample_bytree': 0.9805432708293775}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'learning_rate': FloatDistribution(high=0.3, log=False, low=0.01, step=None), 'n_estimators': IntDistribution(high=1000, log=False, low=100, step=1), 'subsample': FloatDistribution(high=1.0, log=False, low=0.6, step=None), 'colsample_bytree': FloatDistribution(high=1.0, log=False, low=0.6, step=None)}, trial_id=36, value=None)


In [10]:
# Train final model with best params
best_params = study.best_trial.params
best_params.update({"use_label_encoder": False, "eval_metric": "logloss", "random_state": CFG.seed})

In [11]:
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(**best_params))
])

In [12]:
# Final Model & Predictions
final_model.fit(X, y)
y_pred_test = final_model.predict(X_test)
submission = pd.DataFrame({"id": test["id"], CFG.target: label_encoder.inverse_transform(y_pred_test)})
submission.to_csv("submission.csv", index=False)

In [13]:
submission

,id,Personality
0,18524,Extrovert
1,18525,Introvert
2,18526,Extrovert
3,18527,Extrovert
4,18528,Introvert
...,...,...
6170,24694,Extrovert
6171,24695,Introvert
6172,24696,Extrovert
6173,24697,Extrovert
